# Global Deforestation Analysis (1990–2025)
## Notebook 07: Interactive Visualizations & Geospatial Cartography

### Overview
This notebook presents publication-quality, interactive visualization components developed for pairing with the Streamlit dashboard:
1. Dynamic World Choropleth Cartography (Plotly & GeoPandas integration).
2. Country-Level Deep-Dive Subplots with Dual-Axis Trajectories.
3. Multi-Driver Interactive Sunburst & Stacked Area Visualizations.
4. Bivariate Comparative Diagnostics.


In [1]:
import os, sys
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import geopandas as gpd

PROJ_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJ_ROOT)
from src.utils import load_processed_data, THEME, PLOTLY_LAYOUT
from src.charts import (
    create_world_choropleth,
    create_global_trend_chart,
    create_drivers_pie_chart,
    create_commodity_breakdown_chart,
    create_country_deep_dive_chart
)

data = load_processed_data()
df_panel = data["country_panel"]
df_global = data["global_summary"]
df_drivers = data["drivers"]
df_comm = data["commodities"]
print("All processed datasets ready for interactive visualization rendering.")


All processed datasets ready for interactive visualization rendering.


### 1. Interactive World Map: Forest Share (% of Land Area)
Explore national forest density worldwide.


In [2]:
fig_share = create_world_choropleth(df_panel, year=2020, metric="Forest_Share_pct")
fig_share.show()


### 2. Country Deep-Dive: Brazil (BRA)
Dual-axis visualization illustrating total forest area trajectory against annual net deforestation rate.


In [3]:
fig_brazil = create_country_deep_dive_chart(df_panel, iso3="BRA")
fig_brazil.show()


### 3. Country Deep-Dive: Democratic Republic of the Congo (COD)
The Congo Basin is the second largest tropical rainforest on Earth.


In [4]:
fig_drc = create_country_deep_dive_chart(df_panel, iso3="COD")
fig_drc.show()


### 4. Interactive Sunburst: Drivers of Forest Loss
Hierarchical representation of direct drivers across continents.


In [5]:
# Prepare sunburst dataset
sun_data = []
driver_cols = ["Permanent agriculture", "Logging", "Wildfire", "Shifting cultivation", "Settlements and infrastructure", "Mining and energy industry"]

df_drv_recent = df_drivers[df_drivers["Year"] == 2023]
for cont in ["Africa", "Asia", "Europe", "North America", "South America", "Oceania"]:
    sub = df_drv_recent[df_drv_recent["Continent"] == cont]
    for d in driver_cols:
        val = sub[d].sum()
        if val > 0:
            sun_data.append({"Continent": cont, "Driver": d, "Area_ha": val})

df_sun = pd.DataFrame(sun_data)

fig_sun = px.sunburst(
    df_sun,
    path=["Continent", "Driver"],
    values="Area_ha",
    title="Continental Distribution of Direct Drivers of Tree Cover Loss (2023)",
    color="Driver",
    color_discrete_map={
        "Permanent agriculture": "#dc2626",
        "Logging": "#059669",
        "Wildfire": "#ea580c",
        "Shifting cultivation": "#eab308",
        "Settlements and infrastructure": "#6366f1",
        "Mining and energy industry": "#475569"
    }
)
fig_sun.update_layout(**PLOTLY_LAYOUT)
fig_sun.show()


### 5. Geospatial Boundaries with GeoPandas
Load and verify GeoPandas polygon boundaries from `countries.geojson`.


In [6]:
geojson_path = os.path.join(PROJ_ROOT, "data", "raw", "countries.geojson")
gdf_world = gpd.read_file(geojson_path)
print(f"GeoDataFrame loaded: {len(gdf_world)} administrative polygons.")
gdf_world.head(2)[["name", "ISO3166-1-Alpha-3", "geometry"]]


GeoDataFrame loaded: 258 administrative polygons.


,name,ISO3166-1-Alpha-3,geometry
0,Indonesia,IDN,"MULTIPOLYGON (((117.70361 4.16342, 117.70361 4..."
1,Malaysia,MYS,"MULTIPOLYGON (((117.70361 4.16342, 117.69711 4..."


### Visual Cartography Conclusions
The interactive visual architecture enables seamless exploration from macro-continental aggregates down to national dynamics. The visual components are now ready for immediate integration into the Streamlit web dashboard.
